# Script 5 — Simulação de cenários com Gemini
Este notebook consome os artefatos do Script 4 e produz cenários executivos para as empresas âncora.

A lógica é deliberadamente dividida em duas camadas:

1. **Camada quantitativa (Python)**: consolida projeções, intervalos, resíduos e risco.
2. **Camada qualitativa (Gemini)**: interpreta o cenário e devolve uma leitura executiva estruturada em JSON.

A API do Gemini é usada aqui como **gerador de narrativa e estrutura**, não como motor numérico principal.

Sim. Se você quiser construir algo próximo do que aparece na literatura de "AI for Financial Scenario Analysis" e "LLM-based Decision Support Systems", eu recomendo não colocar toda a lógica dentro de um único script Python.

Para o Script 5 ficar realmente profissional, auditável e reproduzível, eu criaria alguns arquivos auxiliares.

Estrutura mínima:

```text
script_5_gemini_cenarios/
│
├── script_5.py
│
├── prompts/
│   ├── sistema.txt
│   ├── cenario_base.txt
│   ├── cenario_otimista.txt
│   ├── cenario_pessimista.txt
│   └── analise_riscos.txt
│
├── config/
│   ├── parametros.yaml
│   └── setores.yaml
│
├── outputs/
│   ├── cenarios_empresa.csv
│   ├── cenarios_empresa.json
│   └── relatorios/
│
└── utils/
    ├── gemini_client.py
    ├── scenario_builder.py
    ├── parser.py
    └── report_generator.py
```

Eu considero esses arquivos praticamente obrigatórios:

1. sistema.txt

Prompt mestre.

Exemplo:

```text
Você é um analista financeiro sênior especializado em valuation,
planejamento estratégico e análise de cenários.

Utilize exclusivamente os dados fornecidos.

Não invente informações externas.

Responda em JSON válido.
```

Isso evita prompt gigante dentro do código.

---

2. parametros.yaml

Controla o comportamento sem alterar código.

```yaml
gemini:
  model: gemini-2.5-pro

cenarios:
  otimista: 15
  base: 0
  pessimista: -15

projecoes:
  anos: 5
```

---

3. setores.yaml

A literatura mostra que cenários dependem muito do setor.

Exemplo:

```yaml
bancos:
  drivers:
    - inadimplencia
    - selic
    - spread

energia:
  drivers:
    - demanda
    - capex
    - regulacao

varejo:
  drivers:
    - inflacao
    - consumo
    - renda
```

O Gemini recebe contexto diferente para cada setor.

---

4. scenario_builder.py

Provavelmente o componente mais importante.

Transforma:

```python
receita_projetada
ebitda_projetado
lucro_projetado
bpa_projetado
```

em:

```python
payload_llm = {
    "empresa": "PETR4",
    "setor": "energia",
    "forecast_2027": {...},
    "forecast_2028": {...}
}
```

---

5. parser.py

Fundamental.

LLMs ocasionalmente retornam JSON malformado.

Esse módulo:

```python
def parse_gemini_response():
    ...
```

corrige e valida.

---

6. report_generator.py

Gera:

```text
- Cenário Base
- Cenário Otimista
- Cenário Pessimista
- Principais Riscos
- Principais Oportunidades
- Impacto esperado no EBITDA
- Impacto esperado no BPA
```

e salva em:

```text
JSON
CSV
Markdown
```

---

O que eu NÃO recomendo:

```python
prompt = f"""
...
"""
```

com milhares de linhas dentro do notebook.

Isso vira um pesadelo para manutenção.

---

Para o seu TCC/projeto de portfólio eu faria algo ainda mais robusto:

```text
Script 1 → Coleta CVM

Script 2 → Feature Engineering

Script 3 → Treinamento

Script 4 → Projeções futuras

Script 5 → Simulação com Gemini

Script 6 → Relatório Executivo Automatizado
```

Nesse desenho, o Script 5 não faz previsões. Ele consome as previsões do Script 4 e produz inteligência estratégica: cenários, riscos, oportunidades, explicações e recomendações para investidores ou gestores. Isso é exatamente onde um LLM agrega mais valor do que um modelo estatístico tradicional.


In [ ]:
import os
import json
import pickle
import warnings
import logging
from logging.handlers import RotatingFileHandler
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# Configuração geral
# ---------------------------------------------------------------------
PASTA_SAIDA = Path("outputs")
PASTA_LOGS = PASTA_SAIDA / "logs"
PASTA_LOGS.mkdir(parents=True, exist_ok=True)
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
ARQ_LOG = PASTA_LOGS / f"script5_{RUN_ID}.log"
ARQ_JSONL = PASTA_LOGS / f"script5_{RUN_ID}.jsonl"

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
GEMINI_TEMPERATURE = float(os.getenv("GEMINI_TEMPERATURE", "1.0"))

print(f"RUN_ID={RUN_ID}")
print(f"Modelo Gemini={GEMINI_MODEL} | temperature={GEMINI_TEMPERATURE}")
print(f"Saída em: {PASTA_SAIDA.resolve()}")

# ---------------------------------------------------------------------
# Logger
# ---------------------------------------------------------------------
logger = logging.getLogger("script5")
logger.setLevel(logging.INFO)
logger.handlers.clear()

fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s")

h_console = logging.StreamHandler()
h_console.setFormatter(fmt)

h_file = RotatingFileHandler(ARQ_LOG, maxBytes=2_000_000, backupCount=5, encoding="utf-8")
h_file.setFormatter(fmt)

logger.addHandler(h_console)
logger.addHandler(h_file)

def registrar_evento(etapa: str, msg: str, nivel: str = "info", **kwargs):
    payload = {
        "ts": datetime.now().isoformat(timespec="seconds"),
        "run_id": RUN_ID,
        "etapa": etapa,
        "nivel": nivel,
        "msg": msg,
        **kwargs,
    }

    linha = json.dumps(payload, ensure_ascii=False, default=str)
    with open(ARQ_JSONL, "a", encoding="utf-8") as f:
        f.write(linha + "\n")

    txt = f"{etapa} | {msg}"
    if kwargs:
        txt += " | " + " | ".join(f"{k}={v}" for k, v in kwargs.items())

    if nivel.lower() == "warning":
        logger.warning(txt)
    elif nivel.lower() == "error":
        logger.error(txt)
    else:
        logger.info(txt)

registrar_evento("bootstrap", "script 5 iniciado", modelo=GEMINI_MODEL, temperature=GEMINI_TEMPERATURE)

In [ ]:
# ---------------------------------------------------------------------
# Funções de carregamento e checagem de artefatos
# ---------------------------------------------------------------------
def carregar_df(path: Path, desc: str, required_cols: Optional[List[str]] = None) -> pd.DataFrame:
    if not path.exists():
        registrar_evento("load", f"{desc} ausente", nivel="warning", arquivo=str(path.name))
        return pd.DataFrame()

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, encoding="utf-8-sig")

    if df.empty:
        registrar_evento("load", f"{desc} vazio", nivel="warning", arquivo=str(path.name))
    else:
        registrar_evento("load", f"{desc} carregado", arquivo=str(path.name), linhas=len(df), colunas=len(df.columns))

    if required_cols:
        faltantes = [c for c in required_cols if c not in df.columns]
        if faltantes:
            registrar_evento(
                "load",
                f"{desc} sem colunas críticas",
                nivel="warning",
                faltantes=",".join(faltantes),
            )
    return df


def carregar_pickle(path: Path, desc: str):
    if not path.exists():
        registrar_evento("load", f"{desc} ausente", nivel="warning", arquivo=str(path.name))
        return None
    with open(path, "rb") as f:
        obj = pickle.load(f)
    registrar_evento("load", f"{desc} carregado", arquivo=str(path.name))
    return obj


# ---------------------------------------------------------------------
# Artefatos do pipeline anterior
# ---------------------------------------------------------------------
dataset = carregar_df(PASTA_SAIDA / "dataset_cvm_consolidado.parquet", "dataset consolidado", ["NOME_CIA", "SETOR", "ANO"])
df_stress = carregar_df(PASTA_SAIDA / "analise_estresse_mc.csv", "Monte Carlo", ["empresa", "setor", "target", "p5", "p50", "p95"])
df_residuos = carregar_df(PASTA_SAIDA / "residuos_detalhados.csv", "resíduos detalhados", ["Target", "Variavel", "Modo", "SMAPE"])
df_conformal = carregar_df(PASTA_SAIDA / "conformal_intervals.csv", "intervalos conformais", ["Target", "Variavel", "ModoEscala", "q_hat"])
df_risco = carregar_df(PASTA_SAIDA / "score_risco_dataset.parquet", "score de risco", ["CNPJ_CIA", "score_risco", "classe_risco"])
zscore_por_empresa = carregar_pickle(PASTA_SAIDA / "zscore_por_empresa.pkl", "z-score por empresa")
melhores_modelos = carregar_pickle(PASTA_SAIDA / "melhores_modelos_v4.pkl", "melhores modelos")

if df_stress.empty:
    raise FileNotFoundError("O Script 5 depende de analise_estresse_mc.csv gerado pelo Script 4.")

# colunas auxiliares
for df in [df_stress, df_residuos, df_conformal, df_risco]:
    if not df.empty:
        for c in df.columns:
            if c.startswith("Unnamed:"):
                df.drop(columns=[c], inplace=True)

print("\nArquivos carregados com sucesso.")
print(f"df_stress: {df_stress.shape}")
print(f"df_residuos: {df_residuos.shape}")
print(f"df_conformal: {df_conformal.shape}")
print(f"df_risco: {df_risco.shape}")

In [ ]:
# ---------------------------------------------------------------------
# Empresas âncora
# ---------------------------------------------------------------------
ANCORAS = {
    "Petrobras": {"keyword": "PETROBRAS", "setor": "Petróleo"},
    "Equatorial Energia": {"keyword": "EQUATORIAL", "setor": "Energia"},
    "Magazine Luiza": {"keyword": "MAGAZINE LUIZA", "setor": "Varejo"},
    "Vale": {"keyword": "VALE", "setor": "Commodities"},
    "WEG": {"keyword": "WEG", "setor": "Tecnologia"},
}

def localizar_empresa(dataset: pd.DataFrame, keyword: str, setor: Optional[str] = None) -> Optional[str]:
    if dataset.empty or "NOME_CIA" not in dataset.columns:
        return None
    base = dataset.copy()
    base["NOME_CIA"] = base["NOME_CIA"].astype(str)
    cand = base[base["NOME_CIA"].str.contains(keyword, case=False, na=False)]
    if setor and "SETOR" in cand.columns:
        cand2 = cand[cand["SETOR"].astype(str).str.contains(setor, case=False, na=False)]
        if not cand2.empty:
            cand = cand2
    if cand.empty:
        return None
    if "ANO" in cand.columns:
        cand = cand.sort_values("ANO")
    return str(cand.iloc[-1]["NOME_CIA"])


empresas_ancora = {}
for nome_curto, info in ANCORAS.items():
    nome_real = localizar_empresa(dataset, info["keyword"], info["setor"])
    if nome_real is None and not df_stress.empty:
        subset = df_stress[df_stress["setor"].astype(str).str.contains(info["setor"], case=False, na=False)]
        if not subset.empty:
            nome_real = str(subset.sort_values(["empresa"]).iloc[0]["empresa"])
    empresas_ancora[nome_curto] = {"nome": nome_real, **info}

print("Empresas âncora resolvidas:")
for k, v in empresas_ancora.items():
    print(f"  - {k}: {v['nome']} | {v['setor']}")

In [ ]:
# ---------------------------------------------------------------------
# Consolidação numérica do cenário
# ---------------------------------------------------------------------
MAPA_VARIAVEL = {
    "DRE_3.01": "Receita Líquida",
    "DRE_3.11": "Lucro Líquido",
    "EBITDA": "EBITDA",
    "DFC_MI_6.01": "FCO",
    "BPA_1": "Ativo Total",
    "BPA_1.01": "Ativo Circulante",
    "BPP_2": "Passivo Total",
    "BPP_2.01": "Passivo Circulante",
    "BPP_2.03": "Patrimônio Líquido",
}

TARGETS_FOCO = [
    "TARGET_DRE_3.01_DFP",
    "TARGET_DRE_3.11_DFP",
    "TARGET_EBITDA_DFP",
    "TARGET_DFC_MI_6.01_DFP",
    "TARGET_BPA_1_DFP",
    "TARGET_BPA_1.01_DFP",
    "TARGET_BPP_2_DFP",
    "TARGET_BPP_2.01_DFP",
    "TARGET_BPP_2.03_DFP",
]

def base_variavel(target: str) -> str:
    s = target.replace("TARGET_", "").replace("_DFP", "")
    s = s.rsplit("_ITR", 1)[0]
    s = s.rsplit("_DFP", 1)[0]
    return s

def nome_variavel(target: str) -> str:
    return MAPA_VARIAVEL.get(base_variavel(target), base_variavel(target))

def resumo_empresa(df_stress: pd.DataFrame, empresa: str) -> pd.DataFrame:
    sub = df_stress[df_stress["empresa"] == empresa].copy()
    if sub.empty:
        return sub
    sub["variavel"] = sub["target"].map(nome_variavel)
    sub["base"] = sub["target"].map(base_variavel)
    return sub


def juntar_residuos(df_base: pd.DataFrame, empresa: str) -> pd.DataFrame:
    if df_base.empty:
        return pd.DataFrame()
    sub = df_base[df_base["CNPJ_CIA"] == empresa].copy() if "CNPJ_CIA" in df_base.columns else df_base.copy()
    return sub


def estimar_contexto_risco(empresa_nome: str, setor: str) -> Dict:
    ctx = {
        "score_risco": None,
        "classe_risco": None,
        "altman_z_ultimo": None,
        "zona_altman": None,
    }

    if not df_risco.empty and "NOME_CIA" in df_risco.columns:
        sub = df_risco[df_risco["NOME_CIA"].astype(str).str.contains(str(empresa_nome), case=False, na=False)].copy()
        if sub.empty and "SETOR" in df_risco.columns:
            sub = df_risco[df_risco["SETOR"].astype(str).str.contains(str(setor), case=False, na=False)].copy()
        if not sub.empty:
            if "ANO" in sub.columns:
                sub = sub.sort_values("ANO")
            row = sub.iloc[-1]
            ctx["score_risco"] = float(row["score_risco"]) if pd.notna(row.get("score_risco")) else None
            ctx["classe_risco"] = str(row["classe_risco"]) if pd.notna(row.get("classe_risco")) else None
            ctx["altman_z_ultimo"] = float(row["altman_z_pp"]) if pd.notna(row.get("altman_z_pp")) else None
            ctx["zona_altman"] = str(row["zona_altman"]) if pd.notna(row.get("zona_altman")) else None

    if isinstance(zscore_por_empresa, dict):
        for k, v in zscore_por_empresa.items():
            nome = str(v.get("nome", ""))
            if empresa_nome and empresa_nome.lower() in nome.lower():
                ctx["altman_z_ultimo"] = v.get("z_ultimo", ctx["altman_z_ultimo"])
                ctx["zona_altman"] = v.get("zona_ultimo", ctx["zona_altman"])
                break

    return ctx


def copiar_colunas_completas(df: pd.DataFrame, cols: List[str], fill_value=np.nan) -> pd.DataFrame:
    out = pd.DataFrame(columns=cols)
    if df.empty:
        return out
    for c in cols:
        if c in df.columns:
            out[c] = df[c]
        else:
            out[c] = fill_value
    return out


def construir_pacote_empresa(empresa_label: str, empresa_nome: str, setor: str) -> Dict:
    sub = resumo_empresa(df_stress, empresa_nome)
    if sub.empty:
        return {}

    if "target" in sub.columns:
        sub = sub[sub["target"].isin(TARGETS_FOCO)].copy() if any(sub["target"].isin(TARGETS_FOCO)) else sub.copy()

    if "target" not in sub.columns:
        return {}

    if not df_residuos.empty:
        sub = sub.merge(
            df_residuos[["Target", "Modo", "SMAPE", "residuo_media", "residuo_mediana", "superestimacao_pct", "score_alinhamento", "ratio_mediano"]],
            left_on="target",
            right_on="Target",
            how="left"
        )
    else:
        for c in ["Modo", "SMAPE", "residuo_media", "residuo_mediana", "superestimacao_pct", "score_alinhamento", "ratio_mediano"]:
            sub[c] = np.nan

    if not df_conformal.empty:
        conf_cols = ["Target", "CoberturaReal", "LarguraMedia", "ModoEscala", "SchemaLen", "SchemaHash"]
        conf_cols = [c for c in conf_cols if c in df_conformal.columns]
        if conf_cols:
            sub = sub.merge(df_conformal[conf_cols], left_on="target", right_on="Target", how="left", suffixes=("", "_conf"))

    sub["bias"] = pd.to_numeric(sub.get("residuo_mediana", np.nan), errors="coerce")
    sub["spread"] = pd.to_numeric(sub["p95"], errors="coerce") - pd.to_numeric(sub["p5"], errors="coerce")
    sub["spread"] = sub["spread"].fillna(0.0)
    sub["score_risco"] = np.nan
    sub["classe_risco"] = np.nan
    sub["altman_z_ultimo"] = np.nan
    sub["zona_altman"] = np.nan
    risco = estimar_contexto_risco(empresa_nome, setor)
    for k, v in risco.items():
        sub[k] = v

    # normalização simples do risco para ajustar os cenários
    score_risco = risco.get("score_risco")
    risco_mult = 1.0
    if pd.notna(score_risco):
        risco_mult += min(max((float(score_risco) - 50.0) / 100.0, -0.15), 0.35)
    if str(risco.get("zona_altman", "")).lower().find("insolv") >= 0:
        risco_mult += 0.10

    cenarios = []
    for nome_cenario, mult in [
        ("Base_Calibrado", 1.00),
        ("Stress_Setorial", 1.35),
        ("Recuperacao_Operacional", 0.85),
    ]:
        rows = []
        for _, r in sub.iterrows():
            center = float(r["p50"])
            p5 = float(r["p5"])
            p95 = float(r["p95"])
            spread = max(float(r["spread"]), abs(center) * 0.03)
            bias = float(r["bias"]) if pd.notna(r["bias"]) else 0.0

            # o cenário não "substitui" a projeção; ele ajusta o envelope
            adj = center + 0.15 * bias * (1 if nome_cenario != "Stress_Setorial" else -1)
            half = 0.50 * spread * mult * risco_mult

            low = adj - half
            high = adj + half

            rows.append({
                "target": r["target"],
                "variavel": r["variavel"],
                "base_p50": center,
                "p5": p5,
                "p95": p95,
                "projeção_ajustada": adj,
                "faixa_baixa": low,
                "faixa_alta": high,
                "SMAPE": float(r["SMAPE"]) if pd.notna(r.get("SMAPE")) else np.nan,
                "residuo_mediana": float(r["residuo_mediana"]) if pd.notna(r.get("residuo_mediana")) else np.nan,
                "classe_risco": r.get("classe_risco"),
                "zona_altman": r.get("zona_altman"),
                "score_risco": r.get("score_risco"),
            })

        cenarios.append({
            "nome": nome_cenario,
            "multiplicador": mult,
            "descricao": {
                "Base_Calibrado": "Cenário central, com pequena correção pelo viés histórico.",
                "Stress_Setorial": "Cenário adverso, ampliando a incerteza e privilegiando leitura conservadora.",
                "Recuperacao_Operacional": "Cenário de melhora operacional, com leitura mais favorável do envelope.",
            }[nome_cenario],
            "tabela": pd.DataFrame(rows),
        })

    return {
        "empresa_label": empresa_label,
        "empresa_nome": empresa_nome,
        "setor": setor,
        "risco": risco,
        "dados": sub,
        "cenarios": cenarios,
    }

print("Funções de consolidação numérica prontas.")

In [ ]:
# ---------------------------------------------------------------------
# Gemini — saída estruturada em JSON
# ---------------------------------------------------------------------
GEMINI_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "empresa": {"type": "string"},
        "setor": {"type": "string"},
        "ano_base": {"type": "integer"},
        "contexto": {"type": "string"},
        "cenarios": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "nome": {"type": "string"},
                    "diagnostico": {"type": "string"},
                    "impactos_chave": {"type": "array", "items": {"type": "string"}},
                    "riscos": {"type": "array", "items": {"type": "string"}},
                    "oportunidades": {"type": "array", "items": {"type": "string"}},
                    "recomendacoes": {"type": "array", "items": {"type": "string"}},
                    "sensibilidade": {"type": "string"},
                    "confianca": {"type": "number"},
                },
                "required": ["nome", "diagnostico", "impactos_chave", "riscos", "oportunidades", "recomendacoes", "sensibilidade", "confianca"],
                "additionalProperties": False,
            },
        },
        "conclusao": {"type": "string"},
        "alertas_modelo": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["empresa", "setor", "ano_base", "contexto", "cenarios", "conclusao", "alertas_modelo"],
    "additionalProperties": False,
}

def montar_prompt(company_pack: Dict) -> str:
    empresa = company_pack["empresa_nome"]
    setor = company_pack["setor"]
    risco = company_pack["risco"]
    rows = []

    for c in company_pack["cenarios"]:
        tabela = c["tabela"].copy()
        if tabela.empty:
            continue
        # reduz o volume: foco nos 7 maiores drivers mais fáceis de ler
        tabela = tabela.sort_values(["SMAPE"], ascending=False).head(7)
        rows.append({
            "nome": c["nome"],
            "descricao": c["descricao"],
            "indicadores": tabela[[
                "target", "variavel", "base_p50", "p5", "p95", "projeção_ajustada", "faixa_baixa", "faixa_alta", "residuo_mediana", "SMAPE"
            ]].round(4).to_dict(orient="records")
        })

    contexto = {
        "empresa": empresa,
        "setor": setor,
        "risco": risco,
        "cenarios": rows,
        "instrucoes": [
            "Use somente os números fornecidos.",
            "Não invente métricas novas.",
            "Se um cenário estiver muito incerto, declare explicitamente a limitação.",
            "Escreva em português do Brasil, tom executivo e objetivo.",
        ]
    }

    return (
        "Você é um analista financeiro sênior. "
        "Sua tarefa é interpretar cenários financeiros de uma empresa brasileira de capital aberto. "
        "A resposta deve ser um JSON válido aderente ao schema solicitado. "
        "Não use markdown, não use listas fora do JSON e não invente números.\n\n"
        f"DADOS:\n{json.dumps(contexto, ensure_ascii=False)}"
    )


def chamar_gemini_json(prompt: str, schema: Dict, model: str = GEMINI_MODEL, temperature: float = GEMINI_TEMPERATURE) -> Dict:
    """
    Prioriza o SDK novo (google-genai). Faz fallback para o SDK legado se necessário.
    """
    # SDK novo
    try:
        from google import genai
        client = genai.Client(api_key=GEMINI_API_KEY)
        resp = client.models.generate_content(
            model=model,
            contents=prompt,
            config={
                "temperature": temperature,
                "response_format": {
                    "text": {
                        "mime_type": "application/json",
                        "schema": schema,
                    }
                },
            },
        )
        txt = getattr(resp, "text", "") or ""
        return json.loads(txt)
    except Exception as e_new:
        registrar_evento("gemini", "falha no SDK novo, tentando fallback", nivel="warning", erro=str(e_new))
        # fallback legado
        try:
            import google.generativeai as genai_legacy
            genai_legacy.configure(api_key=GEMINI_API_KEY)
            modelo = genai_legacy.GenerativeModel(model)
            resp = modelo.generate_content(prompt)
            txt = getattr(resp, "text", "") or ""
            return json.loads(txt)
        except Exception as e_old:
            raise RuntimeError(f"Gemini falhou em ambos SDKs: novo={e_new} | legado={e_old}") from e_old

print("Wrapper Gemini pronto.")

In [ ]:
# ---------------------------------------------------------------------
# Execução dos cenários
# ---------------------------------------------------------------------
resultados_numericos = []
resultados_llm = []
falhas_execucao = []

# se o dataset veio com ano, tenta usar o último ano como referência
ANO_BASE = int(dataset["ANO"].max()) if not dataset.empty and "ANO" in dataset.columns else None

for empresa_label, info in empresas_ancora.items():
    empresa_nome = info["nome"]
    setor = info["setor"]

    if not empresa_nome:
        registrar_evento("exec", f"empresa âncora não resolvida", nivel="warning", empresa=empresa_label, setor=setor)
        continue

    pack = construir_pacote_empresa(empresa_label, empresa_nome, setor)
    if not pack:
        registrar_evento("exec", f"sem dados suficientes para construir pacote", nivel="warning", empresa=empresa_nome, setor=setor)
        continue

    print(f"\n{'='*100}")
    print(f"EMPRESA: {empresa_nome} | SETOR: {setor}")
    print(f"{'='*100}")

    # consolidação numérica primeiro
    for c in pack["cenarios"]:
        tabela = c["tabela"].copy()
        if tabela.empty:
            continue
        tabela["empresa_label"] = empresa_label
        tabela["empresa"] = empresa_nome
        tabela["setor"] = setor
        tabela["cenario"] = c["nome"]
        resultados_numericos.append(tabela)

    # Gemini interpreta o pacote inteiro da companhia
    try:
        prompt = montar_prompt(pack)
        payload = chamar_gemini_json(prompt, GEMINI_JSON_SCHEMA)
        payload["empresa_label"] = empresa_label
        payload["timestamp"] = datetime.now().isoformat(timespec="seconds")
        resultados_llm.append(payload)

        registrar_evento(
            "gemini",
            "análise gerada com sucesso",
            empresa=empresa_nome,
            setor=setor,
            cenarios=len(payload.get("cenarios", [])),
        )

        print(f"Gemini OK | cenários interpretados: {len(payload.get('cenarios', []))}")
        print(payload.get("conclusao", "")[:900])

    except Exception as e:
        falhas_execucao.append({
            "empresa_label": empresa_label,
            "empresa": empresa_nome,
            "setor": setor,
            "erro": str(e),
        })
        registrar_evento("gemini", "falha ao gerar análise", nivel="error", empresa=empresa_nome, setor=setor, erro=str(e))
        print(f"⚠️  Gemini falhou para {empresa_nome}: {e}")

print("\nExecução concluída.")
print(f"Resultados numéricos: {len(resultados_numericos)} blocos")
print(f"Resultados Gemini: {len(resultados_llm)} relatórios")
print(f"Falhas: {len(falhas_execucao)}")

In [ ]:
# ---------------------------------------------------------------------
# Persistência e resumo final
# ---------------------------------------------------------------------
if resultados_numericos:
    df_num = pd.concat(resultados_numericos, ignore_index=True)
    df_num.to_csv(PASTA_SAIDA / "resultados_cenarios_numericos.csv", index=False, encoding="utf-8-sig")
    df_num.to_parquet(PASTA_SAIDA / "resultados_cenarios_numericos.parquet", index=False)
else:
    df_num = pd.DataFrame()

if resultados_llm:
    df_llm = pd.json_normalize(resultados_llm)
    df_llm.to_csv(PASTA_SAIDA / "resultados_cenarios_gemini.csv", index=False, encoding="utf-8-sig")
    df_llm.to_parquet(PASTA_SAIDA / "resultados_cenarios_gemini.parquet", index=False)
else:
    df_llm = pd.DataFrame()

if falhas_execucao:
    df_falhas = pd.DataFrame(falhas_execucao)
    df_falhas.to_csv(PASTA_SAIDA / "falhas_cenarios_gemini.csv", index=False, encoding="utf-8-sig")
else:
    df_falhas = pd.DataFrame()

if not df_num.empty:
    resumo = (
        df_num.groupby(["empresa", "setor", "cenario", "variavel"], as_index=False)
        .agg(
            base_p50=("base_p50", "mean"),
            faixa_baixa=("faixa_baixa", "mean"),
            faixa_alta=("faixa_alta", "mean"),
            SMAPE=("SMAPE", "mean"),
            residuo_mediana=("residuo_mediana", "mean"),
        )
    )
    resumo.to_csv(PASTA_SAIDA / "resumo_cenarios_numericos.csv", index=False, encoding="utf-8-sig")
else:
    resumo = pd.DataFrame()

print("\n" + "="*100)
print("RESUMO FINAL — SCRIPT 5")
print("="*100)
print(f"Numérico: {len(df_num):,} linhas")
print(f"Gemini:   {len(df_llm):,} relatórios")
print(f"Falhas:   {len(df_falhas):,}")

if not df_llm.empty:
    cols = [c for c in ["empresa", "setor", "ano_base", "conclusao"] if c in df_llm.columns]
    print("\nAmostra das conclusões:")
    print(df_llm[cols].head(3).to_string(index=False))

registrar_evento(
    "final",
    "script 5 concluído",
    num_linhas=int(len(df_num)),
    relatorios=int(len(df_llm)),
    falhas=int(len(df_falhas))
)